# First steps with sentence embeddings

An embedding model turns a text into a vector, so that two texts with close meanings get close vectors. The model of these labs: [Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B), multilingual, 1024 dimensions, one of the best open models of its size.

1. Compare a dozen sentences
2. Search three weeks of press, with and without an instruction
3. Search in another language
4. Cosine, dot product, Euclidean distance
5. Truncate the vectors (Matryoshka)

This notebook runs faster with a GPU: *Runtime > Change runtime type > T4 GPU*. Without one, it still runs: the only computation too long for a CPU, the vectors of exercise 2's 3,950 passages, runs on 8 passages there, then its result computed on a GPU is loaded instead.

## Data and model

The passages of the 1914 French press (see the previous lab), their vectors computed on a GPU (for a run without one), and the model, loaded in half precision on a GPU: half the memory, faster.

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/passages.parquet
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/french-press-1914-passages/lab-results/month-embeddings.npy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
# On a CPU, float32: half precisions (float16, and bfloat16, the published weights' one,
# which we'd get by default) are very slow there.
DTYPE = torch.float16 if GPU else torch.float32
print("GPU" if GPU else "No GPU: the long computations will load their result")

model = SentenceTransformer(
  "Qwen/Qwen3-Embedding-0.6B", device=DEVICE, model_kwargs={"dtype": DTYPE}
)
# The model accepts 32,000 tokens: we cut at 512, enough for our passages, otherwise
# the longest ones fill the GPU's memory.
model.max_seq_length = 512

## 1. Comparing a dozen sentences

Encode the sentences of `SENTENCES` with `model.encode_document`, compute the matrix of their similarities with `model.similarity`, and show it with `plt.imshow` (the sentences as axis labels). Which pairs get the best scores? Find two surprises.

**Useful documentation**

- [`matplotlib.pyplot.imshow`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html)
- [`Axes.set_xticks`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.set_xticks.html)

In [ ]:
SENTENCES = [
  "The VPN keeps disconnecting",
  "I can't connect to the network from home",
  "Mon VPN se déconnecte sans arrêt",
  "I received a suspicious email",
  "A fraudulent message arrived in my inbox",
  "The printer stopped printing",
  "My screen stays black",
  "How do I book time off?",
  "When does the payslip arrive?",
  "The server works",
  "The server doesn't work",
  "The canteen closes at 2 pm",
]

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `model.encode_document(SENTENCES)`, then `model.similarity(vectors, vectors).numpy()`: the square matrix of similarities
- `fig, ax = plt.subplots(figsize=(9, 8))`, `image = ax.imshow(similarities)`, then `fig.colorbar(image)` for the color legend
- `ax.set_xticks(range(n), SENTENCES, rotation=90)`: the positions, then the labels (turned by 90°); likewise for `ax.set_yticks`, without rotation

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
vectors = ...  # the sentences' vectors
similarities = model.similarity(..., ...).numpy()  # the matrix of similarities

fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(...)
ax.set_xticks(range(len(SENTENCES)), SENTENCES, rotation=90)
ax.set_yticks(range(len(SENTENCES)), ...)
fig.colorbar(image)
plt.show()
```

### Solution

In [ ]:
vectors = model.encode_document(SENTENCES)
similarities = model.similarity(vectors, vectors).numpy()

fig, ax = plt.subplots(figsize=(9, 8))
image = ax.imshow(similarities, cmap="viridis")
ax.set_xticks(range(len(SENTENCES)), SENTENCES, rotation=90)
ax.set_yticks(range(len(SENTENCES)), SENTENCES)
fig.colorbar(image)
plt.show()

# - A sentence and its French translation are among the closest pairs (0.86): the model
#   is multilingual.
# - But "The server works" and "The server doesn't work", of opposite meanings, are closer
#   still (0.88, the closest pair): embeddings capture the topic much better than a
#   negation.
# - Two unrelated sentences don't drop to 0: only comparing scores makes sense.

## 2. Searching three weeks of press

`month` holds the passages published between July 20 and August 10, 1914 (next cell). Encode their texts, `texts`, with `model.encode_document` (without a GPU, `texts` holds the first 8 only: the next exercise loads the vectors of all of them, computed on a GPU).

Then write a function `search(query, k=5, instruction=True)` that prints the date, the score and the beginning of the `k` passages closest to the query. With `instruction=True`, encode the query with `model.encode_query`, which puts an instruction before it ("*Given a web search query, retrieve relevant passages that answer the query*"); otherwise with `model.encode`, with no instruction.

**Useful documentation**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
passages = pd.read_parquet("passages.parquet")
# The passages from July 20 to August 10, 1914, numbered from 0: a passage's row in month
# is also its vector's position.
month = passages[passages.date.between("1914-07-20", "1914-08-10")].reset_index(
  drop=True
)
print(len(month), "passages")
# Every text on a GPU; without one, the first 8 (about a minute).
texts = month.text.tolist() if GPU else month.text[:8].tolist()

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `model.encode_document(texts, batch_size=32, show_progress_bar=True)`: the passages' vectors, in batches of 32, with a progress bar
- `model.encode_query(query)` with the instruction, `model.encode(query)` without
- `model.similarity(query_vector, month_vectors)[0].numpy()`: one score per passage
- `np.argsort(-scores)[:k]`: the positions of the `k` best (we sort the opposite scores to get the best first)
- `f"{month.date[i]:%d/%m} {scores[i]:.3f} {month.text[i][:150]}"`: the date, the score and the start of passage `i`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
month_vectors = model.encode_document(
  ..., batch_size=32, show_progress_bar=True
)  # the texts: texts


def search(query: str, k: int = 5, instruction: bool = True) -> None:
  query_vector = model.encode_query(query) if instruction else model.encode(...)
  scores = model.similarity(query_vector, month_vectors)[
    0
  ].numpy()  # one score per passage
  for i in ...:  # the positions of the k best scores
    print(f"{month.date[i]:%d/%m} {scores[i]:.3f} {month.text[i][:150]}")
```

### Solution

In [ ]:
month_vectors = model.encode_document(texts, batch_size=32, show_progress_bar=True)


def search(query: str, k: int = 5, instruction: bool = True) -> None:
  query_vector = model.encode_query(query) if instruction else model.encode(query)
  scores = model.similarity(query_vector, month_vectors)[0].numpy()
  for i in np.argsort(-scores)[:k]:
    print(f"{month.date[i]:%d/%m} {scores[i]:.3f} {month.text[i][:150]}")

### Three searches

Search, with and without the instruction (the corpus is in French; the model searches across languages):

- the assassination of Jean Jaurès
- the general mobilization
- the verdict of Mme Caillaux's trial

Are the results relevant? Does the instruction change anything?

In [ ]:
# Without a GPU, the previous exercise encoded 8 passages only: we load the vectors of
# the 3,950, computed on a GPU by the same code.
if not GPU:
  month_vectors = np.load("month-embeddings.npy").astype(np.float32)
print(month_vectors.shape)

In [ ]:
# Your code here

#### Additional hints

The calls and parameters to use:

- `search(query, instruction=True)`, then `instruction=False`, for each of the three queries

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
for query in (
  "the assassination of Jean Jaurès",
  "the general mobilization",
  "the verdict of Mme Caillaux's trial",
):
  for instruction in (True, False):
    print(f"--- {query} ({'with' if instruction else 'without'} instruction)")
    search(...)
```

#### Solution

In [ ]:
for query in (
  "the assassination of Jean Jaurès",
  "the general mobilization",
  "the verdict of Mme Caillaux's trial",
):
  for instruction in (True, False):
    print(f"--- {query} ({'with' if instruction else 'without'} instruction)")
    search(query, instruction=instruction)

# Query by query:
# - Jaurès: with and without the instruction, the five passages report the assassination
#   or the tributes that followed; the instruction only changes their order.
# - The general mobilization: with the instruction, the five passages are about a
#   mobilization (Belgian, Austrian, French, Russian); without it, four, and one about
#   enlisting for the war. The beginning shown doesn't always hold the relevant part: the
#   first passage's Belgian mobilization comes further on.
# - The verdict (the acquittal, on the evening of July 28): without the instruction, the
#   report of the trial's end, which announces it (29/07), comes 2nd; with it, it leaves
#   the top 5.
# The instruction helps on one query and hurts on another: don't conclude from three
# examples, a set of judged queries decides (next lab). And its scores don't compare with
# those of an encoding without it: compare the scores of one query with each other.

## 3. Searching in another language

The corpus is in French and the model multilingual: a query and its translation should find the same passages. Write a function `top(query, k=10)` that returns the positions of the `k` passages closest to the query, encoded with the instruction. For each of the previous exercise's three queries and its French version (« *l'assassinat de Jean Jaurès* », « *la mobilisation générale* », « *le verdict du procès de Mme Caillaux* »), measure the share of passages their top 10s have in common. And why not in another language you know?

Success criterion: the Jaurès query and its translation have the same top 10 passages (100%). Are the other pairs as close? For the one furthest from it, show its first 15 passages in both languages with `search`, and compare their scores.

**Useful documentation**

- [Sets (`set`)](https://docs.python.org/3/library/stdtypes.html#set-types-set-frozenset)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `top`: as in `search`, `model.encode_query(query)` then `model.similarity(query_vector, month_vectors)[0].numpy()`, and `np.argsort(-scores)[:k]` to return
- `set(top(query)) & set(top(translation))`: the passages both rankings share; their number divided by 10 gives the share
- `search(query, k=15)`: the first 15 passages, with their scores

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
def top(query: str, k: int = 10) -> np.ndarray:
  query_vector = model.encode_query(query)
  scores = model.similarity(query_vector, month_vectors)[0].numpy()
  return ...  # the positions of the k best scores


for query, translation in (
  ("the assassination of Jean Jaurès", "l'assassinat de Jean Jaurès"),
  ("the general mobilization", "la mobilisation générale"),
  ("the verdict of Mme Caillaux's trial", "le verdict du procès de Mme Caillaux"),
):
  common = ...  # the passages both rankings share
  print(f"{len(common) / 10:4.0%} of passages in common: {query} / {translation}")

search(..., k=15)  # the pair furthest from 100%, in English
search(..., k=15)  # then in French
```

### Solution

In [ ]:
def top(query: str, k: int = 10) -> np.ndarray:
  query_vector = model.encode_query(query)
  scores = model.similarity(query_vector, month_vectors)[0].numpy()
  return np.argsort(-scores)[:k]


for query, translation in (
  ("the assassination of Jean Jaurès", "l'assassinat de Jean Jaurès"),
  ("the general mobilization", "la mobilisation générale"),
  ("the verdict of Mme Caillaux's trial", "le verdict du procès de Mme Caillaux"),
  ("the assassination of Jean Jaurès", "die Ermordung von Jean Jaurès"),
):
  common = set(top(query)) & set(top(translation))
  print(f"{len(common) / 10:4.0%} of passages in common: {query} / {translation}")

search("the general mobilization", k=15)
search("la mobilisation générale", k=15)

# Jaurès 100%, Caillaux 90%, the mobilization 60% (and Jaurès in German, 70%): a query and
# its translation land almost in the same place, the model is multilingual. For the
# mobilization, the first three passages are the same, in the same order; the differences
# come lower down, where the scores follow each other closely (less than 0.01 from the 8th
# to the 14th passage for the French query): a trifle reorders them. A proper name
# (Jaurès, Caillaux) anchors the query in both languages; the mobilization has many
# passages almost as good.

## 4. Cosine, dot product, Euclidean distance

Compute the norm of the vectors in `month_vectors` (`np.linalg.norm`). Then, for the query "the general mobilization", rank the passages by cosine, dot product and Euclidean distance (closest first): do the three rankings have the same top 10? Why? When would that no longer be the case?

**Useful documentation**

- [`numpy.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)
- [`numpy.ndarray.astype`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html)
- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `month_vectors.astype(np.float32)`: on a GPU, the model is in half precision; we compute in `float32`
- `np.linalg.norm(vectors, axis=1)`: each row's norm
- dot product: `vectors @ query_vector`; cosine: the dot product divided by both vectors' norms; Euclidean distance: `np.linalg.norm(vectors - query_vector, axis=1)`
- `np.argsort(-scores)[:10]` for the dot product and the cosine (largest first), `np.argsort(distance)[:10]` for the distance (smallest first)

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
vectors = month_vectors.astype(np.float32)
print(np.linalg.norm(vectors, axis=1)[:5])

query_vector = model.encode_query("the general mobilization").astype(np.float32)
dot = vectors @ query_vector
cosine = dot / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(...))
euclidean = np.linalg.norm(..., axis=1)
rankings = {
  "cosine": np.argsort(-cosine)[:10],
  "dot product": np.argsort(...)[:10],
  "Euclidean": np.argsort(...)[:10],
}
for name, ranking in rankings.items():
  print(f"{name:12}", ranking)
```

### Solution

In [ ]:
vectors = month_vectors.astype(np.float32)
print(np.linalg.norm(vectors, axis=1)[:5])

query_vector = model.encode_query("the general mobilization").astype(np.float32)
dot = vectors @ query_vector
cosine = dot / (np.linalg.norm(vectors, axis=1) * np.linalg.norm(query_vector))
euclidean = np.linalg.norm(vectors - query_vector, axis=1)
rankings = {
  "cosine": np.argsort(-cosine)[:10],
  "dot product": np.argsort(-dot)[:10],
  "Euclidean": np.argsort(euclidean)[:10],
}
for name, ranking in rankings.items():
  print(f"{name:12}", ranking)

# The model normalizes its vectors (norm 1): the dot product is then the cosine, and
# |u - v|² = 2 - 2 cos(u, v), so the Euclidean distance ranks in the same order. With
# vectors of different norms, the dot product would favor the longest ones.

## 5. Truncating the vectors (Matryoshka)

The model is trained so that the **first** dimensions carry most of the meaning. Truncate the passages' and queries' vectors to 512, 256, 128 and 64 dimensions (then renormalize them), and measure, for 200 random passages used as queries, the share of their 10 nearest neighbors (in 1024 dimensions) found again after truncation. Do the same with binary quantization, which keeps the 1024 dimensions but only their sign (one bit each). How many bytes does a vector take at each size, and how much memory would 1 million passages take?

**Useful documentation**

- [Matryoshka](https://sbert.net/examples/sentence_transformer/training/matryoshka/README.html)
- [`numpy.linalg.norm`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.norm.html)
- [`numpy.random.Generator.choice`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html)
- [`numpy.where`](https://numpy.org/doc/stable/reference/generated/numpy.where.html)

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `x[:, :dims]` keeps the first `dims` columns; renormalize by dividing by `np.linalg.norm(x, axis=1, keepdims=True)` (`keepdims=True`: one norm per row, in a shape that can divide the matrix)
- `np.random.default_rng(0).choice(len(vectors), 200, replace=False)`: 200 random positions, without replacement
- the 10 nearest neighbors: `x[queries] @ x.T`, with `-np.inf` for the passage itself (which isn't its own neighbor), then `np.argsort(-scores, axis=1)[:, :10]`
- the share found: the intersection of the neighbors before and after truncation, divided by 10, averaged over the queries
- `np.where(vectors > 0, 1.0, -1.0)`: the sign of each dimension; these vectors all have the same norm, so the dot product ranks them like the cosine, with no renormalization
- the memory: `dims * 4` bytes per `float32` vector, `1024 / 8` for one bit per dimension (once the signs are packed into bits), times 1 million
- the model can also truncate its own vectors, with `truncate_dim` when loading it, without renormalizing them

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
def truncate(x: np.ndarray, dims: int) -> np.ndarray:
  x = x[:, :dims]
  return x / np.linalg.norm(..., axis=1, keepdims=True)


rng = np.random.default_rng(0)
queries = rng.choice(len(vectors), 200, replace=False)


def neighbours(x: np.ndarray) -> np.ndarray:
  scores = x[queries] @ x.T
  scores[np.arange(len(queries)), queries] = ...  # a passage isn't its own neighbor
  return np.argsort(..., axis=1)[:, :10]


def compare(name: str, x: np.ndarray, nbytes: float) -> None:
  found = neighbours(x)
  overlap = np.mean(
    [... for a, b in zip(reference, found)]
  )  # the share of neighbors found again
  print(
    f"{name:17}: {overlap:4.0%} of neighbors found, {nbytes:4.0f} bytes per vector,"
    f" {nbytes * 1e6 / 1e9:.1f} GB per million passages"
  )


reference = neighbours(vectors)
compare("1024 dimensions", vectors, 1024 * 4)
for dims in (512, 256, 128, 64):
  compare(f"{dims} dimensions", truncate(vectors, dims), ...)  # bytes per vector
compare("1024 bits (signs)", ..., 1024 / 8)  # the sign of each dimension
```

### Solution

In [ ]:
def truncate(x: np.ndarray, dims: int) -> np.ndarray:
  x = x[:, :dims]
  return x / np.linalg.norm(x, axis=1, keepdims=True)


rng = np.random.default_rng(0)
queries = rng.choice(len(vectors), 200, replace=False)


def neighbours(x: np.ndarray) -> np.ndarray:
  scores = x[queries] @ x.T
  scores[np.arange(len(queries)), queries] = -np.inf  # a passage isn't its own neighbor
  return np.argsort(-scores, axis=1)[:, :10]


def compare(name: str, x: np.ndarray, nbytes: float) -> None:
  found = neighbours(x)
  overlap = np.mean([len(set(a) & set(b)) / 10 for a, b in zip(reference, found)])
  print(
    f"{name:17}: {overlap:4.0%} of neighbors found, {nbytes:4.0f} bytes per vector,"
    f" {nbytes * 1e6 / 1e9:.1f} GB per million passages"
  )


reference = neighbours(vectors)
compare("1024 dimensions", vectors, 1024 * 4)
for dims in (512, 256, 128, 64):
  compare(f"{dims} dimensions", truncate(vectors, dims), dims * 4)
compare("1024 bits (signs)", np.where(vectors > 0, 1.0, -1.0), 1024 / 8)

# The overlap measures stability, not quality: two different neighbors can be just as
# good. Only a set of judged queries measures it (next lab).
# The signs of the 1024 dimensions find almost as many neighbors (64%) as 256 dimensions
# in float32 (67%), for 128 bytes per vector instead of 1,024: 8 times smaller, and 32
# times smaller than the whole vector (4,096 bytes). Hence the practice: search with the
# binary vectors, then rerank the best ones with the real vectors.